# E2 BioBART Sentence No-context Fine-tuning


## 1.Imports and Configuration


In [ ]:
from __future__ import annotations

import gc
import os
import random
import time
from collections import Counter
from pathlib import Path
from typing import Any

LOCAL_CACHE_DIR = Path.cwd() / ".cache"
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(LOCAL_CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(LOCAL_CACHE_DIR))
os.environ.setdefault("HF_HOME", str(LOCAL_CACHE_DIR / "huggingface"))
# Disable hf_transfer unless the package is explicitly installed.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

try:
    import evaluate
except ImportError:
    evaluate = None

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())



In [7]:
SEED = 42
MODEL_NAME = "GanjinZero/biobart-base"
MODEL_CANDIDATES = [
    MODEL_NAME,
    "facebook/bart-base",
]

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "sentence_no_context"
TRAIN_PATH = DATA_DIR / "train_clean.csv"
VAL_PATH = DATA_DIR / "val_clean.csv"
TEST_PATH = DATA_DIR / "test_clean.csv"

OUTPUT_DIR = PROJECT_ROOT / "models" / "biobart_sentence_no_context"
RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "biobart_sentence_no_context_predictions.csv"
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16_supported = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
fp16_enabled = False
bf16_enabled = bf16_supported

print(f"Parameters are fixed and logged.")


Parameters are fixed and logged.


## 2.Dataset


In [9]:
REQUIRED_COLUMNS = ["pair_id", "sent_id", "label", "complex", "simple"]

def load_clean_split(path: Path, split_name: str, allow_empty_simple: bool = True) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} file: {path}")

    df = pd.read_csv(path)

    missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{split_name} is missing columns: {missing_columns}")

    df = df[REQUIRED_COLUMNS].copy()

    for column in ["complex", "simple", "label"]:
        df[column] = df[column].fillna("").astype(str).str.strip()

    df = df[df["complex"].ne("")].reset_index(drop=True)

    if not allow_empty_simple:
        df = df[df["simple"].ne("")].reset_index(drop=True)

    return df


train_df = load_clean_split(TRAIN_PATH, "train", allow_empty_simple=True)
val_df = load_clean_split(VAL_PATH, "validation", allow_empty_simple=True)
test_df = load_clean_split(TEST_PATH, "test", allow_empty_simple=True)

print(f"Loaded train: {len(train_df):,} rows from {TRAIN_PATH.relative_to(PROJECT_ROOT)}")
print(f"Loaded validation: {len(val_df):,} rows from {VAL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Loaded test: {len(test_df):,} rows from {TEST_PATH.relative_to(PROJECT_ROOT)}")

print("Empty targets:")
print(f"train: {train_df['simple'].eq('').sum():,}")
print(f"validation: {val_df['simple'].eq('').sum():,}")
print(f"test: {test_df['simple'].eq('').sum():,}")

Loaded train: 6,742 rows from data/sentence_no_context/train_clean.csv
Loaded validation: 984 rows from data/sentence_no_context/val_clean.csv
Loaded test: 892 rows from data/sentence_no_context/test_clean.csv
Empty targets:
train: 0
validation: 0
test: 0


## 3.Data Inspection



In [10]:
display(train_df.head())

def word_count(text: str) -> int:
    return len(str(text).split())

length_frames = []
for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    length_frames.append(
        pd.DataFrame(
            {
                "split": split_name,
                "complex_words": df["complex"].map(word_count),
                "simple_words": df["simple"].map(word_count),
            }
        )
    )
length_df = pd.concat(length_frames, ignore_index=True)

length_stats = (
    length_df.groupby("split")[["complex_words", "simple_words"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)
display(length_stats)

print("Train label distribution:")
display(train_df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))

max_source_length = 256
max_target_length = 128
print(f"max_source_length = {max_source_length}")
print(f"max_target_length = {max_target_length}")

,pair_id,sent_id,label,complex,simple
0,CD012936,0,rephrase,Three studies (146 participants) met our selec...,Three studies involving 146 participants were ...
1,CD012936,3,split,The three studies assessed palliative care as ...,All three studies compared palliative care del...
2,CD012936,4,rephrase,One of the three studies included participants...,The third study (Ne-PAL) included participants...
3,CD012936,7,rephrase,The three included studies did not assess the ...,"The included studies did not assess fatigue, c..."
4,CD012936,8,ignore,We did not find any trial that compared differ...,We did not find studies that compared differen...


complex_words                                                     \
                   count   mean    std  min   50%   90%   95%    99%    max   
split                                                                         
test               892.0  24.66  16.43  4.0  22.0  41.0  50.0  82.45  223.0   
train             6742.0  24.86  15.66  1.0  21.5  43.0  53.0  80.00  186.0   
validation         984.0  24.41  15.97  4.0  21.0  41.7  53.0  78.34  197.0   

           simple_words                                                      
                  count   mean    std  min   50%   90%    95%    99%    max  
split                                                                        
test              892.0  23.49  13.14  4.0  22.0  40.0  47.45  70.00  144.0  
train            6742.0  23.51  12.96  3.0  21.0  40.0  48.00  66.00  145.0  
validation        984.0  22.99  12.79  4.0  20.0  38.0  44.85  72.17  117.0

Train label distribution:


,label,count
0,rephrase,5239
1,ignore,982
2,split,521


max_source_length = 256
max_target_length = 128


## 3.Tokenization


In [ ]:
PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""

def build_prompt(complex_sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=str(complex_sentence).strip())


def preprocess_examples(examples: dict[str, list[Any]]) -> dict[str, Any]:
    inputs = [build_prompt(text) for text in examples["complex"]]
    targets = [str(text).strip() for text in examples["simple"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_source_length,
        truncation=True,
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True,
    )["input_ids"]

    labels = [
        [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in label]
        for label in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs


sample_prompt = build_prompt(train_df.loc[0, "complex"])
print(sample_prompt)
print("Target:", train_df.loc[0, "simple"])



## 4.Dataset Creation



In [14]:
train_dataset_raw = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset_raw = Dataset.from_pandas(val_df, preserve_index=False)
test_dataset_raw = Dataset.from_pandas(test_df, preserve_index=False)

remove_columns = train_dataset_raw.column_names
train_dataset = train_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
val_dataset = val_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)
test_dataset = test_dataset_raw.map(preprocess_examples, batched=True, remove_columns=remove_columns)

# Keep Python lists here. DataCollatorForSeq2Seq performs dynamic padding and tensor conversion.
# Pre-formatting as torch can trigger slow list-of-ndarray tensor warnings for labels.

print(train_dataset)
print(val_dataset)
print(test_dataset)

Map: 100%|██████████| 892/892 [00:00<00:00, 7142.23 examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 6742
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 984
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 892
})


In [16]:
def inspect_tokenized_labels(dataset: Dataset, name: str, n: int = 2) -> None:
    print(f"{name} label sanity check")
    for idx in range(min(n, len(dataset))):
        labels = dataset[idx]["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        active_labels = [token_id for token_id in labels if token_id != -100]
        print(f"  example {idx}: active target tokens = {len(active_labels)}")
        print("  decoded target:", tokenizer.decode(active_labels, skip_special_tokens=True))
    empty_count = 0
    for row in dataset:
        labels = row["labels"]
        if hasattr(labels, "tolist"):
            labels = labels.tolist()
        if not any(token_id != -100 for token_id in labels):
            empty_count += 1
    print(f"  empty targets: {empty_count} / {len(dataset)}")
    if empty_count:
        raise ValueError(f"{name} has empty tokenized targets; check the simple column and preprocessing.")

inspect_tokenized_labels(train_dataset, "train")
inspect_tokenized_labels(val_dataset, "validation")

train label sanity check
  example 0: active target tokens = 13
  decoded target: Three studies involving 146 participants were included in this review.
  example 1: active target tokens = 51
  decoded target: All three studies compared palliative care delivered in home visits versus usual care for people with MS. Two studies included only participants with MS. In all three studies, interventions focused on assessment and management of symptoms and end-of-life planning.
  empty targets: 0 / 6742
validation label sanity check
  example 0: active target tokens = 68
  decoded target: We found 16 randomised controlled trials (studies where treatments are decided at random; these usually give the most reliable evidence about treatment effects) comparing glucocorticoids around the time of embryo implantation versus no glucocorticoids or placebo (dummy treatment), in 2232 couples undergoing IVF/ICSI.
  example 1: active target tokens = 27
  decoded target: Considering the quality of evidence,

## 5.Model Loading



In [17]:
CRITICAL_MISSING_KEYS = {
    "model.encoder.embed_tokens.weight",
    "model.decoder.embed_tokens.weight",
    "lm_head.weight",
}


def load_seq2seq_model(model_candidates: list[str], resolved_tokenizer_model: str) -> tuple[Any, str]:
    ordered_candidates = [resolved_tokenizer_model] + [
        candidate for candidate in model_candidates if candidate != resolved_tokenizer_model
    ]
    errors = []
    for candidate in ordered_candidates:
        try:
            model, loading_info = AutoModelForSeq2SeqLM.from_pretrained(
                candidate,
                output_loading_info=True,
            )
            missing_keys = set(loading_info.get("missing_keys", []))
            critical_missing = sorted(missing_keys & CRITICAL_MISSING_KEYS)
            if critical_missing:
                del model
                gc.collect()
                message = f"critical missing keys: {critical_missing}"
                errors.append(f"{candidate}: {message}")
                print(f"Skipping model {candidate}: {message}")
                continue
            unexpected_keys = loading_info.get("unexpected_keys", [])
            if unexpected_keys:
                print(f"Model {candidate} has unexpected keys: {unexpected_keys[:5]}")
            print(f"Loaded model: {candidate}")
            return model, candidate
        except Exception as exc:
            errors.append(f"{candidate}: {type(exc).__name__}: {exc}")
            print(f"Could not load model {candidate}: {exc}")
    raise RuntimeError("Could not load any model candidate without critical missing keys.\n" + "\n".join(errors))

model, RESOLVED_MODEL_NAME = load_seq2seq_model(MODEL_CANDIDATES, RESOLVED_MODEL_NAME)
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
)

print(f"Resolved model: {RESOLVED_MODEL_NAME}")
print(f"Model loaded on: {next(model.parameters()).device}")

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 19107.78it/s]


Loaded model: GanjinZero/biobart-base
Resolved model: GanjinZero/biobart-base
Model loaded on: cpu


## 6.Training


In [ ]:
def build_training_args() -> Seq2SeqTrainingArguments:
    base_kwargs = dict(
        output_dir=str(OUTPUT_DIR),
        num_train_epochs=5,
        learning_rate=3e-5,
        weight_decay=0.01,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=fp16_enabled,
        bf16=bf16_enabled,
        logging_nan_inf_filter=False,
        report_to="none",
        seed=SEED,
    )
    try:
        return Seq2SeqTrainingArguments(
            evaluation_strategy="epoch",
            **base_kwargs,
        )
    except TypeError:
        return Seq2SeqTrainingArguments(
            eval_strategy="epoch",
            **base_kwargs,
        )

training_args = build_training_args()

def build_trainer() -> Seq2SeqTrainer:
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        return Seq2SeqTrainer(processing_class=tokenizer, **trainer_kwargs)
    except TypeError:
        return Seq2SeqTrainer(tokenizer=tokenizer, **trainer_kwargs)

trainer = build_trainer()
trainer.train()
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))
print(f"Saved best model to: {BEST_MODEL_DIR.relative_to(PROJECT_ROOT)}")

### Training Log



In [ ]:
training_log_df = pd.DataFrame(trainer.state.log_history)
display(training_log_df)

epoch_eval_df = training_log_df[training_log_df["eval_loss"].notna()].copy()
if len(epoch_eval_df):
    display(epoch_eval_df[["epoch", "step", "eval_loss", "eval_runtime"]])
else:
    print("No eval_loss rows found in trainer log history.")

## 7.Predictions




In [ ]:
GENERATION_CONFIG = {
    "max_new_tokens": max_target_length,
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
}

print("Generation config:", GENERATION_CONFIG)

def clean_prediction(text: str) -> str:
    """Remove prompt echoes and generation boilerplate from decoded text."""
    text = re.sub(r"\s+", " ", str(text).strip())
    if not text:
        return ""

    # Decoder-only or poorly adapted seq2seq checkpoints can echo the input prompt.
    prompt_markers = [
        "Simplified sentence:",
        "Rewrite the biomedical sentence for a general audience.",
        "Rewrite this biomedical sentence in simpler language:",
        "Simplify the biomedical sentence.",
        "Sentence:",
    ]
    for marker in prompt_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    prefixes = ["Simplified:", "Answer:", "Prediction:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text).strip()

def generate_batch(sentences: list[str]) -> list[str]:
    input_texts = [build_prompt(sentence) for sentence in sentences]

    inputs = tokenizer(
        input_texts,
        max_length=max_source_length,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **GENERATION_CONFIG,
        )

    decoded = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
    return [clean_prediction(text) for text in decoded]


def generate_predictions(df: pd.DataFrame, batch_size: int = 8) -> pd.DataFrame:
    model.eval()
    predictions = []
    sentences = df["complex"].fillna("").astype(str).tolist()

    for start in tqdm(range(0, len(sentences), batch_size), desc="Generating"):
        batch_sentences = sentences[start : start + batch_size]
        try:
            batch_predictions = generate_batch(batch_sentences)
        except Exception as exc:
            print(f"Generation failed for rows {start}-{start + len(batch_sentences) - 1}: {exc}")
            batch_predictions = [""] * len(batch_sentences)

        predictions.extend(batch_predictions)

    output_df = df[["pair_id", "sent_id", "label", "complex", "simple"]].copy()
    output_df["prediction"] = predictions
    return output_df


prediction_df = generate_predictions(test_df, batch_size=8)
prediction_df.to_csv(PREDICTION_PATH, index=False)

print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(prediction_df.head())



## 8.Evaluation

The metrics mirror the FLAN-T5 notebook. SARI evaluates simplification edits, BLEU is reported with sacreBLEU on a `0-100` scale for comparability with the E1 result, and BERTScore measures semantic similarity.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
display(metrics_summary)


## 9.Analysis


In [ ]:
example_columns = ["complex", "simple", "prediction"]
qualitative_examples = prediction_df[example_columns].sample(
    n=min(20, len(prediction_df)),
    random_state=SEED,
)
display(qualitative_examples)

## 10.Final Comparison

This table keeps E0 and E1 baselines beside the computed E2 metrics. BLEU is reported on the sacreBLEU `0-100` scale.

In [ ]:
def metric_value(summary: pd.DataFrame, metric: str) -> float:
    values = summary.loc[summary["metric"].eq(metric), "score"].tolist()
    return float(values[0]) if values else float("nan")

comparison_df = pd.DataFrame(
    [
        {
            "Experiment": "E0",
            "Model": "Llama 3.1 8B zero-shot",
            "SARI": 28.25,
            "BLEU": 3.6,
            "BERTScore F1": 0.897,
        },
        {
            "Experiment": "E1",
            "Model": "FLAN-T5-base fine-tuned",
            "SARI": 26.86,
            "BLEU": 36.57,
            "BERTScore F1": 0.935,
        },
        {
            "Experiment": "E2",
            "Model": f"{RESOLVED_MODEL_NAME} fine-tuned",
            "SARI": metric_value(metrics_summary, "SARI"),
            "BLEU": metric_value(metrics_summary, "BLEU"),
            "BERTScore F1": metric_value(metrics_summary, "BERTScore F1"),
        },
    ]
)
display(comparison_df)